In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 01 — Rolling Window Setup

Partitions the processed Home Credit dataset into chronological time steps. Because running the full pipeline on every feasible monthly pair would be computationally expensive, we materialise a fixed number of window pairs spread approximately evenly over the valid temporal range. This preserves coverage of early, middle, and late periods while keeping downstream training and explanation costs manageable.

**Input:** `data/processed/` (from notebook 00)  
**Output:** `data/windows/window_config.json`

---

**Framework recap (§3.1):**
- Data partitioned into time steps D_{t1}, …, D_{tK} (one per calendar month).
- Training window: W_k = D_{t_{k-L+1}} ∪ … ∪ D_{t_k}  (fixed length L).
- Model A trained on W_k, Model B on W_{k+s}.
- Common evaluation slice: E_{A,B} = D_{t_{k+s+1}} ∪ … ∪ D_{t_{k+s+h}}.

**Parameters used:**
| Parameter | Value | Meaning |
|-----------|-------|---------|
| L | 6 | training window length (months) |
| S | 4 | step between Model A and Model B windows (months) |
| H | 3 | evaluation horizon (months) |
| R | 2 | replicas per window (used by nb 02) |
| K_FRAC | 0.10 | top fraction of eval instances flagged (by max replica-averaged probability of either model) |
| PAIR_STRIDE | 3 | spacing between candidate k values when enumerating pairs |

**Parameter rationale.** The Home Credit span is ~22 months (Jan 2019 – Oct 2020), so monthly granularity gives 22 time steps with comfortable per-step row counts (after the 25% subsample applied in notebook 00). With `L = 6` the training window is 27% of the timeline, matching Homesite's `L = 8 / 29 ≈ 28%` ratio. With `S = 4` the A and B training windows differ in 4 of 6 months (≈67% non-overlap) so consecutive models have meaningful room to differ — the same intent as Homesite's `S/L = 5/8`. `H = 3` matches Homesite's evaluation horizon. `PAIR_STRIDE = 3` together with the constraints `k ≥ L − 1` and `k + S + H ≤ K − 1` admits exactly four window pairs spread evenly across the timeline, matching the Homesite pair count.

**Note on the April 2020 volume cliff.** The Kaggle data ships with a steep dip in application volume during April 2020 (≈10% of the surrounding months), most plausibly a COVID-19 underwriting freeze. After the 25% subsample this leaves ~800 applications in that month. We keep the month as its own time step and document the dip; the pair whose evaluation slice spans Feb–Apr 2020 will therefore probe a window with a real data-drift event present, which is informative for the temporal-stability methodology rather than a defect.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

WORKSPACE = Path('/content/drive/MyDrive/Thesis/HomeCredit_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'
WIN_DIR.mkdir(parents=True, exist_ok=True)  # create the output directory if it does not exist yet

# ── Experiment parameters ────────────────────────────────────────────────
L = 6           # training window length (time steps = months)
S = 4           # step between Model A and Model B windows
H = 3           # evaluation horizon (time steps)
R = 2           # replicas per window
K_FRAC = 0.10   # top-K fraction of eval instances to flag (by max(p_hat_A, p_hat_B))
PAIR_STRIDE = 3 # only every PAIR_STRIDE-th candidate k is materialised as a pair

print(f'Parameters: L={L}, S={S}, H={H}, R={R}, K_FRAC={K_FRAC}, PAIR_STRIDE={PAIR_STRIDE}')

## 1. Load processed data

In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet')
Y    = np.load(PROC_DIR / 'Y.npy')
meta = pd.read_parquet(PROC_DIR / 'meta.parquet')

# reset_index gives a clean 0-based integer index; storing it as 'row_idx'
# keeps a stable positional reference so rows can be located with .iloc on
# X and Y even after groupby / merge operations reshape the DataFrame.
meta = meta.reset_index(drop=True)
meta['row_idx'] = meta.index

print(f'X: {X.shape}, Y: {Y.shape}')
print(f'Date range: {meta["date_decision"].min()} → {meta["date_decision"].max()}')

assert len(X) == len(Y) == len(meta), 'X, Y, and meta have inconsistent lengths.'
assert 'date_decision' in meta.columns, 'meta.parquet must contain a date_decision column.'
assert pd.api.types.is_datetime64_any_dtype(meta['date_decision']), 'date_decision must be datetime-like.'
assert not meta['date_decision'].isna().any(), 'date_decision contains missing values.'
assert meta['date_decision'].is_monotonic_increasing, (
    'date_decision is not monotonic non-decreasing; preprocessing should have sorted by it.'
)

## 2. Define time steps (calendar months)

In [ ]:
# dt.to_period('M') maps each date to its calendar-month period;
# .dt.to_timestamp() converts that period back to the first-of-month
# anchor so all rows in the same calendar month share an identical 'month' value.
meta['month'] = meta['date_decision'].dt.to_period('M').dt.to_timestamp()

# Count applications per month to inspect how the data is distributed over time.
# Highly uneven counts can affect training stability for some windows.
month_counts = meta.groupby('month').size().sort_index()
print('Applications per calendar month:')
print(month_counts.to_string())

fig, ax = plt.subplots(figsize=(13, 3))
month_counts.plot(ax=ax, kind='bar', color='steelblue', width=0.8)
ax.set_title('Applications per calendar month')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
ax.set_xticklabels([m.strftime('%Y-%m') for m in month_counts.index], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(WIN_DIR / 'applications_per_month.png', dpi=120)
plt.show()

# Per-month target summary — useful for spotting drift in the base rate before any modelling.
month_target_summary = (
    meta.assign(y=Y)
        .groupby('month')
        .agg(n=('y', 'size'), positives=('y', 'sum'), pos_rate=('y', 'mean'))
        .sort_index()
)
print('\nApplications and default rate per month:')
print(month_target_summary.to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Sorting gives the canonical temporal ordering t_1 … t_K used throughout the experiment.
# All step indices below are 0-based (t_1 → 0, t_K → K−1).
time_steps = sorted(month_counts.index.tolist())
K = len(time_steps)
print(f'K = {K} time steps')
for i, ts in enumerate(time_steps):
    n = month_counts[ts]
    print(f'  t_{i+1:>2d}: {ts.strftime("%Y-%m")}  ({n:,} applications)')

In [ ]:
# If certain months are very sparse, a single time step may be unreliable
# for training/evaluation. The threshold is set to 1,000 applications per
# month, calibrated to the 25% subsample size in notebook 00 and the heavy
# class imbalance (~3% positives) which makes very small months yield only
# a handful of positives. The April 2020 month is expected to fall below
# this threshold (the COVID volume cliff documented in the header markdown).
MIN_PER_STEP = 1000
sparse_months = [ts for ts, n in month_counts.items() if n < MIN_PER_STEP]
if sparse_months:
    print(f'WARNING: {len(sparse_months)} month(s) have < {MIN_PER_STEP} applications: '
          f'{[ts.strftime("%Y-%m") for ts in sparse_months]}')
    print('These steps remain in the timeline — see the header markdown on the COVID volume cliff.')
else:
    print('All months have sufficient applications.')

# Lookup dict from month timestamp → 0-based integer step index.
# Mapping it onto meta['step'] allows every row to be addressed by its position
# in the temporal sequence rather than a calendar date.
month_to_step = {m: i for i, m in enumerate(time_steps)}
meta['step'] = meta['month'].map(month_to_step)

## 3. Enumerate valid window pairs

A pair (A, B) is valid when:
- k ≥ L − 1  (enough history for training window A)
- k + S + H ≤ K − 1  (evaluation slice fits within the data)

where k is the last time step of window A (0-based). The `PAIR_STRIDE` parameter then keeps only every PAIR_STRIDE-th candidate k, so the materialised pairs are spread across the full temporal range while the downstream compute (notebooks 02–04) stays bounded.

In [ ]:
def get_indices(step_set: set) -> list:
    """Row indices for the given set of time-step indices, sorted chronologically.

    Sorting by date_decision ensures that TimeSeriesSplit in notebooks 02 / 02b
    sees rows in temporal order. row_idx is used as a deterministic tiebreaker
    for applications on the same date.
    """
    return (meta[meta['step'].isin(step_set)]
            .sort_values(['date_decision', 'row_idx'])['row_idx']
            .tolist())


pairs = []

# Iterate over candidate last training step k for Model A, in PAIR_STRIDE jumps.
# Two constraints must both hold:
#   (1) k >= L - 1  →  enough prior steps to fill the L-step training window
#   (2) eval_end = k + S + H < K  →  the evaluation slice fits within the dataset
#
# With K=22, L=6, S=4, H=3, PAIR_STRIDE=3  → 4 pairs (k = 5, 8, 11, 14).
for k in range(L - 1, K, PAIR_STRIDE):       # k = last step of window A (0-based)
    k_b = k + S                              # last step of window B (shifted S steps forward)
    eval_start = k_b + 1                     # first evaluation step (immediately after B's window)
    eval_end   = k_b + H                     # last evaluation step (H steps after eval_start - 1)

    if eval_end >= K:
        break                                # not enough future data for a complete evaluation slice

    # Time-step sets
    steps_A    = set(range(k - L + 1, k + 1))         # W_k:     L steps ending at k
    steps_B    = set(range(k_b - L + 1, k_b + 1))     # W_{k+s}: L steps ending at k+s
    steps_eval = set(range(eval_start, eval_end + 1)) # E_{A,B}: H steps starting after B

    # Row index lists — used directly as positional selectors on X and Y
    idx_A    = get_indices(steps_A)
    idx_B    = get_indices(steps_B)
    idx_eval = get_indices(steps_eval)

    if len(idx_eval) == 0:
        print(f'Skipping pair k={k}: empty evaluation slice')
        continue

    pairs.append({
        'pair_id':           len(pairs),
        'k':                 k,
        'k_b':               k_b,
        'step_label_A':      time_steps[k].strftime('%Y-%m'),
        'step_label_B':      time_steps[k_b].strftime('%Y-%m'),
        'eval_start_label':  time_steps[eval_start].strftime('%Y-%m'),
        'eval_end_label':    time_steps[eval_end].strftime('%Y-%m'),
        'steps_A':           sorted(steps_A),
        'steps_B':           sorted(steps_B),
        'steps_eval':        sorted(steps_eval),
        'idx_A':             idx_A,
        'idx_B':             idx_B,
        'idx_eval':          idx_eval,
        'n_train_A':         len(idx_A),
        'n_train_B':         len(idx_B),
        'n_eval':            len(idx_eval),
    })

print(f'\nTotal valid window pairs: {len(pairs)}')

# Verify step-sorted chronological order of each stored index list.
# Required for TimeSeriesSplit in notebooks 02/02b: rows from earlier steps must
# appear before rows from later steps so each CV fold is a valid out-of-time set.
for p in pairs:
    for label, idx_key in [('A', 'idx_A'), ('B', 'idx_B'), ('eval', 'idx_eval')]:
        steps_seq = meta.loc[p[idx_key], 'step'].values
        assert (steps_seq[:-1] <= steps_seq[1:]).all(), \
            f"Pair {p['pair_id']}: {label} indices are not step-sorted — TSCV ordering broken!"
print('Chronological-order assertions passed.')

In [ ]:
# Summary table: one row per valid pair.
# 'A_window' / 'B_window' show the last month of each model's training window
# (the full window spans the preceding L-1 months as well).
# 'eval_period' is the shared held-out slice used to compare both models.
summary = pd.DataFrame([{
    'pair_id':       p['pair_id'],
    'A_window':      f"{p['step_label_A']}",
    'B_window':      f"{p['step_label_B']}",
    'eval_period':   f"{p['eval_start_label']} → {p['eval_end_label']}",
    'n_train_A':     p['n_train_A'],
    'n_train_B':     p['n_train_B'],
    'n_eval':        p['n_eval'],
    'n_pos_A':       int(Y[p['idx_A']].sum()),
    'n_pos_B':       int(Y[p['idx_B']].sum()),
    'n_pos_eval':    int(Y[p['idx_eval']].sum()),
    'pos_rate_A':    float(Y[p['idx_A']].mean()),
    'pos_rate_B':    float(Y[p['idx_B']].mean()),
    'pos_rate_eval': float(Y[p['idx_eval']].mean()),
} for p in pairs])

with pd.option_context('display.width', 200, 'display.max_columns', None):
    print(summary.to_string(index=False))

# Inject the pos_rate / n_pos fields back into the pair dicts so they end up in the saved JSON.
# Notebook 04's drift analysis reads these directly without needing X/Y in memory.
for p, row in zip(pairs, summary.to_dict('records')):
    p['n_pos_A']       = row['n_pos_A']
    p['n_pos_B']       = row['n_pos_B']
    p['n_pos_eval']    = row['n_pos_eval']
    p['pos_rate_A']    = row['pos_rate_A']
    p['pos_rate_B']    = row['pos_rate_B']
    p['pos_rate_eval'] = row['pos_rate_eval']

empty_eval = summary[summary['n_eval'] == 0]
if not empty_eval.empty:
    print(f'\nWARNING: {len(empty_eval)} pair(s) with empty evaluation slice!')
else:
    print('\nAll evaluation slices are non-empty.')

# Class-balance assertion: every slice must contain both classes, otherwise the
# replica training, threshold-based flagging, and explanation extraction will all break.
for p in pairs:
    for label, idx_key in [('A', 'idx_A'), ('B', 'idx_B'), ('eval', 'idx_eval')]:
        y_slice = Y[p[idx_key]]
        assert len(np.unique(y_slice)) == 2, (
            f"Pair {p['pair_id']}: {label} slice contains only one class."
        )
print('Class-balance assertions passed.')

## 4. Verify temporal separation within each pair

For a fair pairwise comparison, the common evaluation slice must chronologically follow both training windows and must not overlap with either training window.

In [ ]:
for p in pairs:
    # Evaluation steps must not overlap with either training window —
    # overlap would constitute data leakage from the evaluation period into training.
    assert not set(p['steps_eval']) & set(p['steps_A']), f"Pair {p['pair_id']}: eval overlaps A!"
    assert not set(p['steps_eval']) & set(p['steps_B']), f"Pair {p['pair_id']}: eval overlaps B!"
    # The earliest evaluation step must come strictly after B's last training step,
    # guaranteeing temporal ordering: train A → train B → evaluate (no future peeking).
    assert min(p['steps_eval']) > max(p['steps_B']), f"Pair {p['pair_id']}: eval not after B!"

print('Non-overlap checks passed for all pairs.')

## 5. Save window configuration

In [ ]:
# The JSON captures everything needed to reproduce the experiment downstream:
#   'parameters' — scalar hyperparameters (L, S, H, R, K_FRAC, PAIR_STRIDE) and the
#                  ordered list of month labels that define the time axis.
#   'pairs'      — one entry per valid (A, B) pair, each containing:
#                    * integer step indices and human-readable date labels
#                    * pre-computed row index lists (idx_A, idx_B, idx_eval)
#                      that can be used directly with .iloc on X and Y in
#                      subsequent notebooks without re-running this setup
#                    * positive-class counts and rates per slice (consumed by nb 04).
config = {
    'parameters': {
        'L': L, 'S': S, 'H': H, 'R': R, 'K_FRAC': K_FRAC,
        'PAIR_STRIDE': PAIR_STRIDE,
        'K': K,
        'time_unit': 'month',
        'time_steps': [ts.strftime('%Y-%m') for ts in time_steps],
    },
    'pairs': pairs,
}

out_path = WIN_DIR / 'window_config.json'
with open(out_path, 'w') as f:
    json.dump(config, f, indent=2)

size_kb = out_path.stat().st_size / 1024
print(f'Saved {out_path.name} ({size_kb:.0f} KB)')
print(f'{len(pairs)} window pairs, {K} time steps')

In [ ]:
# Visual: training and evaluation set sizes over time.
# Left plot  — Train A vs Train B sample counts per pair.
#              B's window is offset by S steps, so it should overlap heavily
#              with A but include slightly newer data; sizes should be similar.
# Right plot — Evaluation slice size per pair.
#              Large drops would signal data gaps in the later months.
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

pair_ids = [p['pair_id'] for p in pairs]
axes[0].plot(pair_ids, [p['n_train_A'] for p in pairs], 'o-', label='Train A')
axes[0].plot(pair_ids, [p['n_train_B'] for p in pairs], 's--', label='Train B')
axes[0].set_title('Training set sizes')
axes[0].set_xlabel('Window pair')
axes[0].set_ylabel('# instances')
axes[0].set_xticks(pair_ids)
axes[0].legend()

axes[1].bar(pair_ids, [p['n_eval'] for p in pairs], color='darkorange')
axes[1].set_title('Evaluation slice sizes')
axes[1].set_xlabel('Window pair')
axes[1].set_ylabel('# instances')
axes[1].set_xticks(pair_ids)

plt.tight_layout()
plt.savefig(WIN_DIR / 'window_sizes.png', dpi=120)
plt.show()

# Visual: positive class rate per slice across pairs — exposes target drift early.
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(pair_ids, [p['pos_rate_A']    for p in pairs], 'o-',  label='Train A')
ax.plot(pair_ids, [p['pos_rate_B']    for p in pairs], 's--', label='Train B')
ax.plot(pair_ids, [p['pos_rate_eval'] for p in pairs], '^:',  label='Eval')
ax.axhline(float(Y.mean()), color='grey', linestyle='--', linewidth=0.8,
           label=f'Overall ({Y.mean():.2%})')
ax.set_title('Positive class rate per slice')
ax.set_xlabel('Window pair')
ax.set_ylabel('Default rate')
ax.set_xticks(pair_ids)
ax.legend()
plt.tight_layout()
plt.savefig(WIN_DIR / 'pos_rate_per_slice.png', dpi=120)
plt.show()
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Small vertical offset so A/B lines don't overlap visually.
offset = 0.1

for p in pairs:
    pid = p['pair_id']

    # Offset Train A down and Train B up
    ax.plot(p['steps_A'], [pid - offset] * L, 'o-', color='steelblue', alpha=0.6, label='Train A' if pid == 0 else "")
    ax.plot(p['steps_B'], [pid + offset] * L, 's-', color='orange',    alpha=0.6, label='Train B' if pid == 0 else "")

    # Keep Eval centered
    ax.plot(p['steps_eval'], [pid] * H, 'd-', color='crimson', alpha=0.8, label='Eval' if pid == 0 else "")

ax.set_title('Window placement per pair (blue=A, orange=B, red=eval)')
ax.set_xlabel('Time step index (Months)')
ax.set_ylabel('Pair ID')
ax.set_yticks(range(len(pairs)))
ax.set_ylim(-0.5, len(pairs) - 0.5)  # Add some padding
ax.grid(True, axis='x', linestyle='--', alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig(WIN_DIR / 'window_layout.png', dpi=120)
plt.show()